In [1]:
import csv
import random
import pandas as pd
from collections import Counter
import spacy
from spacy.tokens import Doc
import benepar
from nltk.tree import Tree
from spacy import displacy

In [35]:
dev_tsv = "./dev_feature_predict.tsv"
test_tsv = "./test_feature_predict.tsv"
out_tsv = "./dev_test_feature_predict.tsv"

with open(out_tsv, "w", encoding="utf-8") as out:

    #dev
    with open(dev_tsv, "r", encoding="utf-8") as dev:
        for line in dev:
            out.write(line)

    #sentence break between dev and test
    out.write("\n")

    #test
    with open(test_tsv, "r", encoding="utf-8") as test:
        for line in test:
            out.write(line)


In [3]:
combined_tsv = "./dev_test_feature_predict.tsv"

sentences = []
current_sentence = []

with open(combined_tsv, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
        else:
            cols = line.split("\t")
            current_sentence.append(cols)

# catch last sentence if file doesn't end with newline
if current_sentence:
    sentences.append(current_sentence)

In [4]:
false_positives = []
false_negatives = []

for sent_id, sent in enumerate(sentences):
    sentence_text = " ".join(tok[0] for tok in sent)

    for tok_id, tok in enumerate(sent):
        token = tok[0]
        gold = tok[9]      # gold scope label
        pred = tok[10]     # predicted scope label

        # False Positive: predicted scope, gold says no scope
        if pred == "x" and gold == "_":
            false_positives.append({
                "sentence_id": sent_id,
                "token_id": tok_id,
                "token": token,
                "sentence": sentence_text,
                "gold": gold,
                "pred": pred
            })

        # False Negative: missed a scope token
        elif pred == "_" and gold == "x":
            false_negatives.append({
                "sentence_id": sent_id,
                "token_id": tok_id,
                "token": token,
                "sentence": sentence_text,
                "gold": gold,
                "pred": pred
            })

print(f"False Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")


False Positives: 600
False Negatives: 896


In [5]:
import random
from collections import defaultdict

random.seed(42)

#Group FP tokens by sentence
fps_by_sent = defaultdict(list)
for tok in false_positives:
    sent_idx = tok["sentence_id"]
    fps_by_sent[sent_idx].append(tok)

#Group FN tokens by sentence
fns_by_sent = defaultdict(list)
for tok in false_negatives:
    sent_idx = tok["sentence_id"]
    fns_by_sent[sent_idx].append(tok)

#Shuffle sentence IDs
fp_sent_ids = list(fps_by_sent.keys())
fn_sent_ids = list(fns_by_sent.keys())

random.shuffle(fp_sent_ids)
random.shuffle(fn_sent_ids)

#Sample 25 FP tokens, one per sentence
fp_sample = []
for sent_id in fp_sent_ids:
    fp_sample.append(random.choice(fps_by_sent[sent_id]))
    if len(fp_sample) >= 25:
        break

#Sample 25 FN tokens, one per sentence
fn_sample = []
for sent_id in fn_sent_ids:
    fn_sample.append(random.choice(fns_by_sent[sent_id]))
    if len(fn_sample) >= 25:
        break

print(f"Selected {len(fp_sample)} FP tokens and {len(fn_sample)} FN tokens")


Selected 25 FP tokens and 25 FN tokens


In [6]:
for ex in fp_sample[:3]:
    print("FP token:", ex["token"])
    print("Sentence:", ex["sentence"])
    print()

for ex in fn_sample[:3]:
    print("FN token:", ex["token"])
    print("Sentence:", ex["sentence"])
    print()


FP token: cream-laid
Sentence: `` The note is written upon ordinary cream-laid paper without watermark .

FP token: Nothing
Sentence: `` Nothing wonderful in that , surely ? ''

FP token: he
Sentence: The mysterious one could understand English , even if he could not print it .

FN token: you
Sentence: I should not have intruded it upon your attention had you not shown some incredulity the other day .

FN token: cease
Sentence: `` But he would never cease talking of it -- your kindness , sir , and the way in which you brought light into the darkness .

FN token: ,
Sentence: `` ` Well , I do n't know now whether it was pure devilry on the part of this woman , or whether she thought that she could turn me against my wife by encouraging her to misbehave .



In [7]:
import csv

with open("error_analysis_sample.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "error_type",
        "sentence_id",
        "token_id",
        "token",
        "gold",
        "pred",
        "sentence"
    ])

    for ex in fp_sample:
        writer.writerow([
            "FP",
            ex["sentence_id"],
            ex["token_id"],
            ex["token"],
            ex["gold"],
            ex["pred"],
            ex["sentence"]
        ])

    for ex in fn_sample:
        writer.writerow([
            "FN",
            ex["sentence_id"],
            ex["token_id"],
            ex["token"],
            ex["gold"],
            ex["pred"],
            ex["sentence"]
        ])

In [10]:
nlp = spacy.load("en_core_web_md")
nlp.add_pipe("benepar", config={"model": "benepar_en3"})

def parse_sent(sentence):
    """  
    Parse the sentence using Spacy and benepar.
    """

    if type(sentence) == list:
        spacy_doc = Doc(nlp.vocab, words=sentence)
        processed_doc = nlp(spacy_doc)
    else:
        processed_doc = nlp(sentence)
    
    for sent in list(processed_doc.sents):    
        print([(token.text, token.lemma_, token.pos_, token.tag_) for token in sent]) 
        tree = sent._.parse_string
        #display(Tree.fromstring(tree))
        #displacy.render(sent, style="dep", options={"collapse_punct": False})


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [12]:
#Convert TSV sentences to parsed_sentences
parsed_sentences = []

for sent_id, token_list in enumerate(sentences):
    words = [t[0] for t in token_list]  # column 0 = token text
    doc = Doc(nlp.vocab, words=words)
    spacy_doc = nlp(doc)  # parse with spaCy + benepar
    parsed_sentences.append({
        'sent_id': sent_id,
        'original': token_list,
        'spacy_sent': list(spacy_doc)
    })

print(f"Prepared {len(parsed_sentences)} parsed sentences")

Prepared 1934 parsed sentences


In [111]:
def find_multiple_negation_cases_gold(parsed_sentences):
    """
    Identify sentences with multiple negation cues based on the **gold labels** 
    from your dataset (column 7), not a predefined list.
    """

    correct = []
    incorrect = []

    for sent_data in parsed_sentences:
        sent = sent_data['original']
        sent_id = sent_data['sent_id']

        # Extract all negation cues from this sentence using gold cue labels (column 7)
        neg_cue_words = [tok[0] for tok in sent if tok[7] != '_']

        if len(neg_cue_words) < 2:
            continue  # skip sentences without multiple negations

        # Process all tokens in this sentence
        for tok_idx, tok in enumerate(sent):
            gold = tok[9]
            pred = tok[10]

            # Skip tokens not part of scope
            if gold == '_' and pred == '_':
                continue

            case = {
                'sentence_id': sent_id,
                'token_id': tok_idx,
                'token': tok[0],
                'sentence': ' '.join([t[0] for t in sent]),
                'gold': gold,
                'predicted': pred,
                'correct': gold == pred,
                'negation_count': len(neg_cue_words),
                'negation_cues': ', '.join(neg_cue_words)  # comprehensive
            }

            if case['correct']:
                correct.append(case)
            else:
                incorrect.append(case)

    return correct, incorrect


In [112]:
def find_coordination_cases(parsed_sentences):
    """
    Identify coordination scope cases using dependency parsing.
    Tokens involved in coordination are:
    - cc tokens
    - conj tokens
    - heads of conj tokens
    - tokens with conj children
    """

    correct = []
    incorrect = []

    for sent_data in parsed_sentences:
        sent = sent_data['original']
        spacy_sent = sent_data['spacy_sent']
        sent_id = sent_data['sent_id']

        if not spacy_sent:
            continue

        for tok_idx, tok in enumerate(sent):
            if tok_idx >= len(spacy_sent):
                continue

            st = spacy_sent[tok_idx]

            is_coord = (
                st.dep_ in {"cc", "conj"} or
                any(c.dep_ == "conj" for c in st.children) or
                st.head.dep_ == "conj"
            )

            if not is_coord:
                continue

            gold = tok[9]
            pred = tok[10]

            if gold == '_' and pred == '_':
                continue

            case = {
                'sentence_id': sent_id,
                'token_id': tok_idx,
                'token': tok[0],
                'sentence': ' '.join([t[0] for t in sent]),
                'gold': gold,
                'predicted': pred,
                'correct': gold == pred,
                'dep': st.dep_,
                'head': st.head.text
            }

            if case['correct']:
                correct.append(case)
            else:
                incorrect.append(case)

    return correct, incorrect

In [113]:
# Coordination cases
coord_correct, coord_incorrect = find_coordination_cases(parsed_sentences)

# Multiple negation cases
mn_correct, mn_incorrect = find_multiple_negation_cases(parsed_sentences)


In [114]:
def print_pattern_stats(pattern_name, correct, incorrect):
    total = len(correct) + len(incorrect)
    error_rate = len(incorrect) / total if total > 0 else 0
    print(f"\nPattern: {pattern_name}")
    print(f"Total cases: {total}")
    print(f"Correct: {len(correct)}")
    print(f"Incorrect: {len(incorrect)}")
    print(f"Error rate: {error_rate}")

print_pattern_stats("Coordination", coord_correct, coord_incorrect)
print_pattern_stats("Multiple Negation", mn_correct, mn_incorrect)


Pattern: Coordination
Total cases: 627
Correct: 315
Incorrect: 312
Error rate: 0.49760765550239233

Pattern: Multiple Negation
Total cases: 718
Correct: 384
Incorrect: 334
Error rate: 0.46518105849582175


In [115]:
# Coordination
coord_fp = [c for c in coord_incorrect if c['gold'] == '_']
coord_fn = [c for c in coord_incorrect if c['gold'] == 'x']

print("Coordination FP:", len(coord_fp))
print("Coordination FN:", len(coord_fn))

# Multiple Negation
mn_fp = [c for c in mn_incorrect if c['gold'] == '_']
mn_fn = [c for c in mn_incorrect if c['gold'] == 'x']

print("Multiple Negation FP:", len(mn_fp))
print("Multiple Negation FN:", len(mn_fn))


Coordination FP: 135
Coordination FN: 177
Multiple Negation FP: 151
Multiple Negation FN: 183


In [116]:
def show_examples_simple(pattern_name, correct, incorrect, n=3):
    """Show first n unique examples for correct and incorrect predictions"""
    
    print(f"Examples for {pattern_name}:\n")
    
    # Incorrect predictions
    seen_sentences = set()
    print("Incorrect predictions:")
    count = 0
    for case in incorrect:
        sent = case['sentence']
        if sent not in seen_sentences and count < n:
            print(f"{count+1}. Sentence ID {case['sentence_id']}: {sent}")
            print(f"   Token: '{case['token']}' (pos {case['token_id']}), Gold: {case['gold']}, Predicted: {case['predicted']}")
            if 'dep' in case:
                print(f"   Dependency: {case['dep']} (head: {case['head']})")
            if 'negation_count' in case:
                print(f"   Negation count: {case['negation_count']}, Cues: {case['negation_cues']}")
            seen_sentences.add(sent)
            count += 1
    
    # Correct predictions
    seen_sentences = set()
    print("\nCorrect predictions:")
    count = 0
    for case in correct:
        sent = case['sentence']
        if sent not in seen_sentences and count < n:
            print(f"{count+1}. Sentence ID {case['sentence_id']}: {sent}")
            print(f"   Token: '{case['token']}' (pos {case['token_id']}), Gold: {case['gold']}, Predicted: {case['predicted']}")
            seen_sentences.add(sent)
            count += 1

# Example usage:
show_examples_simple("Coordination", coord_correct, coord_incorrect, n=3)
show_examples_simple("Multiple Negation", mn_correct, mn_incorrect, n=3)


Examples for Coordination:

Incorrect predictions:
1. Sentence ID 36: `` I have had a most singular and unpleasant experience , Mr. Holmes , '' said he .
   Token: 'singular' (pos 6), Gold: _, Predicted: x
   Dependency: dobj (head: had)
2. Sentence ID 45: Private detectives are a class with whom I have absolutely no sympathy , but none the less , having heard your name -- ''
   Token: 'are' (pos 2), Gold: x, Predicted: _
   Dependency: ROOT (head: are)
3. Sentence ID 51: But no one can glance at your toilet and attire without seeing that your disturbance dates from the moment of your waking . ''
   Token: 'and' (pos 8), Gold: x, Predicted: _
   Dependency: cc (head: glance)

Correct predictions:
1. Sentence ID 3: He made no remark , but the matter remained in his thoughts , for he stood in front of the fire afterwards with a thoughtful face , smoking his pipe , and casting an occasional glance at the message .
   Token: 'made' (pos 1), Gold: x, Predicted: x
2. Sentence ID 36: `` I hav